Set the environment variable for the Groq API key to authenticate requests.

In [ ]:
from google.colab import userdata
GROQ_API_KEY=userdata.get('GROQ_API_KEY')
EXCHANGERATE_API_KEY=userdata.get('EXCHANGERATE_API_KEY')

Install the necessary Python packages: `langchain-groq`, `langchain-core`, and `requests`.

In [ ]:
!pip install -q langchain-groq==1.0.0 langchain-core==1.0.4 requests==2.32.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.4 which is incompatible.


In [ ]:
!pip show langchain-groq langchain-core requests

Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-core
Version: 1.0.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions
Required-by: langchain, langchain-groq, langchain-text-splitters
---
Name: requests
Version: 2.32.5
Summary: Python HTTP for Humans.
Home-page: https://requests.readthedocs.io
Author: Kenneth Reitz
Author-email: me@kennethreitz.org
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: certifi, charset_normalizer, idna, urllib3
Required-by: bigfram

Import the required modules from LangChain and the `requests` library.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

Create a simple tool using the `@tool` decorator that multiplies two numbers.

In [ ]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

Invoke the `multiply` tool with sample arguments to test its functionality.

In [ ]:
print(multiply.invoke({'a':3, 'b':4}))

12


Display the name of the `multiply` tool.

In [ ]:
multiply.name

'multiply'

Display the description of the `multiply` tool.

In [ ]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

Display the argument schema for the `multiply` tool.

In [ ]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

Create a ChatGroq language model instance with a specified model and temperature.

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0, api_key=GROQ_API_KEY)

Invoke the LLM with a simple greeting to test its response.

In [ ]:
llm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. The instruction: "You are ChatGPT, a large language model trained by OpenAI." There\'s no special instruction. Just respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 72, 'total_tokens': 129, 'completion_time': 0.057062914, 'prompt_time': 0.003773704, 'queue_time': 0.002874049, 'total_time': 0.060836618, 'completion_tokens_details': {'reasoning_tokens': 39}}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_7b8f00bae0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--9ee5ddba-ecca-4630-aa4d-cfe5fb318aa6-0', usage_metadata={'input_tokens': 72, 'output_tokens': 57, 'total_tokens': 129})

Bind the `multiply` tool to the LLM so it can use the tool during conversations.

In [ ]:
llm_with_tools = llm.bind_tools([multiply])

Invoke the LLM with tools enabled using a simple greeting.

In [ ]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm just a bunch of code, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'reasoning_content': 'User says "Hi how are you". We respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 133, 'total_tokens': 186, 'completion_time': 0.057591482, 'prompt_time': 0.006457077, 'queue_time': 0.00336314, 'total_time': 0.064048559, 'completion_tokens_details': {'reasoning_tokens': 13}}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--9787ffa1-3db9-4140-a086-5ecabb1f6d60-0', usage_metadata={'input_tokens': 133, 'output_tokens': 53, 'total_tokens': 186})

Create a `HumanMessage` asking the LLM to multiply two numbers.

In [ ]:
query = HumanMessage('can you multiply 3 with 1000')

Initialize a list of messages to be used in the conversation.

In [ ]:
messages = [query]

Display the current list of messages.

In [ ]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

Invoke the LLM with the prepared messages and tools.

In [ ]:
result = llm_with_tools.invoke(messages)

Append the AI's response to the messages list.

In [ ]:
messages.append(result)

Display the updated list of messages after appending the AI response.

In [ ]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_619123fe-0648-4a66-b3a7-744a2d03ffbd', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 138, 'total_tokens': 174, 'completion_time': 0.035875978, 'prompt_time': 0.006615694, 'queue_time': 0.003091333, 'total_time': 0.042491672, 'completion_tokens_details': {'reasoning_tokens': 9}}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--b4da9b33-bc1d-4f8c-b835-18dce246bb4e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'fc_619123fe-0648-4a66-b3a7-744a2d03ffbd', 'type': 'tool_call'}], usa

Invoke the `multiply` tool using the arguments extracted from the AI's tool call.

In [ ]:
tool_result = multiply.invoke(result.tool_calls[0])

Display the result returned by the `multiply` tool.

In [ ]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='fc_619123fe-0648-4a66-b3a7-744a2d03ffbd')

Append the tool result to the messages list.

In [ ]:
messages.append(tool_result)

Display the final list of messages after all steps.

In [ ]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_619123fe-0648-4a66-b3a7-744a2d03ffbd', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 138, 'total_tokens': 174, 'completion_time': 0.035875978, 'prompt_time': 0.006615694, 'queue_time': 0.003091333, 'total_time': 0.042491672, 'completion_tokens_details': {'reasoning_tokens': 9}}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--b4da9b33-bc1d-4f8c-b835-18dce246bb4e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'fc_619123fe-0648-4a66-b3a7-744a2d03ffbd', 'type': 'tool_call'}], usa

Invoke the LLM with the full conversation and display its final response.

In [ ]:
llm_with_tools.invoke(messages).content

'Sure! 3 × 1000 = **3000**.'

Define two tools: one to fetch the currency conversion factor using an API, and another to convert a value using the conversion rate.

In [ ]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/{EXCHANGERATE_API_KEY}/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


Display the argument schema for the `convert` tool.

In [ ]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

Invoke the `get_conversion_factor` tool with sample currencies to test its output.

In [ ]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1762732801,
 'time_last_update_utc': 'Mon, 10 Nov 2025 00:00:01 +0000',
 'time_next_update_unix': 1762819201,
 'time_next_update_utc': 'Tue, 11 Nov 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 88.7342}

Invoke the `convert` tool with sample values to test its output.

In [ ]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

Bind the currency conversion tools to a new LLM instance.

In [ ]:
# tool binding
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0, api_key=GROQ_API_KEY)

Bind the `get_conversion_factor` and `convert` tools to the LLM.

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

Create a `HumanMessage` asking for the conversion factor and to convert a value between INR and USD.

In [ ]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

Display the current list of messages for the currency conversion task.

In [ ]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

Invoke the LLM with the currency conversion query and tools.

In [ ]:
ai_message = llm_with_tools.invoke(messages)

Append the AI's response to the messages list.

In [ ]:
messages.append(ai_message)

Display the tool calls suggested by the AI in its response.

In [ ]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'fc_5574408f-a21f-4c88-bf99-5b5aa26a7b9d',
  'type': 'tool_call'}]

Iterate through the tool calls, execute them, and update the messages list with the results.

In [ ]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



Display the updated list of messages after executing the tool calls.

In [ ]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function get_conversion_factor to get factor between INR and USD. Then use convert to convert 10 INR to USD. The convert function expects base_currency_value: number. But we need to specify base_currency? The function signature: convert(_:{base_currency_value: number}) => any. It doesn\'t specify target currency. Maybe the context is that the conversion factor is already known? Actually convert likely uses the previously fetched conversion factor. But we need to call get_conversion_factor first. Then call convert with base_currency_value: 10. But we need to pass base_currency? The function signature only has base_currency_value. So maybe the convert function uses the last fetched conversion factor. So we call get_conversion_factor with base_c

Invoke the LLM with the full conversation and display its final response for the currency conversion task.

In [ ]:
llm_with_tools.invoke(messages).content

'**Conversion factor (INR → USD):** 1\u202fINR = **0.01127\u202fUSD**\n\n**10\u202fINR to USD:**\n\n\\[\n10 \\text{\u202fINR} \\times 0.01127 \\frac{\\text{USD}}{\\text{INR}} = 0.1127 \\text{\u202fUSD}\n\\]\n\nSo, 10\u202fINR is approximately **$0.11 USD** (rounded to two decimal places).'